In [1]:
from torch import nn
from collections import OrderedDict

import numpy as np

from matplotlib import pyplot as plt

from xaikd import models, datasets, utils, attributors, logit_modifiers, bases

In [2]:
DEVICE = utils.get_device()
SEED = 1


In [3]:
model = models.get_trained_model("celeba-resnet18-finetunedv1")

In [4]:
dataset = datasets.construct("celeba-attr25")

In [5]:
train_loader, _, val_loader, test_loader = (
    datasets.construct_dataloaders(
        dataset=dataset,
        training_data_ratio=0.01,
        seed=1,
        use_validation_set=True,
    )
)

[use_validation=True]: ratio_train=0.0100, ratio_val=0.2000
==== Dataset Information [use_validation_set=True] ====
> split=train: count=1628
> split=val  : count=32554
> split=test : count=19867


In [6]:
# prepare teacher
teacher_model = nn.Sequential(
    OrderedDict(
        [
            ("base", model),
            (
                "last_layer",
                models.layers.resolve_teacher_last_layer(dataset=dataset),
            ),
        ]
    )
)
teacher_model.eval()
teacher_model.to(DEVICE)

Sequential(
  (base): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_run

In [7]:
logit_mod = logit_modifiers.BinaryLogOddWinning(threshold=0)

arr_logodd, arr_act, arr_ctx, mean_act = attributors.extract_activation_grad(
    model=teacher_model,
    layer="base.layer4",
    dataloader=train_loader,
    logit_modifier=logit_mod,
    device=DEVICE,
    rng=np.random.default_rng(seed=SEED),
)

extract act-grad at layer=base.layer4:   0%|          | 0/26 [00:00<?, ?it/s]

> shape(arr_act)=(1628, 512, 20); shape(arr_logits)=(1628,)


In [8]:
def fit_basis(basis_name):
        
    print(f"fitting basis={basis_name}")
    basis = bases.get_basis(basis_name)
    basis.fit(
        arr_act=arr_act,
        arr_ctx=arr_ctx,
        mean_act=mean_act,
        arr_logodd=arr_logodd,
        logodd_threshold=logit_mod.threshold,
    )
    
    return basis

basis_pca = fit_basis("pca")

basis_prcaposdef = fit_basis("prcaposdef-entropy0.95")

fitting basis=pca
fitting basis=prcaposdef-entropy0.95
[entropy_ratio=0.95]: best_ix=22 best_std=7.5961e-01 best_candidate=0.23
Coefficients: coeff_acca=1.0000e+00, coeff_a=5.9230e-07, coeff_c=6.7534e+06 tr_a=1.1374e+03, tr_c=9.9750e-11
range(eigvals)=[5.1184e-08, 6.8044e-04]


In [9]:
np.sum(basis_pca.scale_factors[:32]), np.sum(basis_prcaposdef.scale_factors[:32])

(827.07605, 823.8392493935936)

In [12]:
arr_act_flattened = utils.flatten_3d_tensor(arr_act)
N, _ = arr_act_flattened.shape

cov_a = arr_act_flattened.T @ arr_act_flattened / N

eigvals_a = np.linalg.eigvalsh(cov_a)

In [15]:
eigvals_a[:32]

array([0.08641922, 0.09675647, 0.09858315, 0.1004518 , 0.10163122,
       0.104012  , 0.10515828, 0.10624332, 0.1081965 , 0.10918807,
       0.10980443, 0.11053182, 0.1119037 , 0.11208831, 0.11331226,
       0.11358224, 0.11434035, 0.11657657, 0.11737889, 0.11849997,
       0.11862756, 0.11921299, 0.12090229, 0.1222621 , 0.12298305,
       0.12306724, 0.12352256, 0.12470835, 0.12615077, 0.12679927,
       0.12713619, 0.12824716], dtype=float32)

In [19]:
U_pca = basis_pca.U


scale = U_pca.T @ cov_a @ U_pca